# Demo: Cross-Patient Transfer Learning for Stimulus Encoders

This is a minimal, exploratory notebook — NOT used to generate the paper's results.
For reproducing the actual results, use the scripts in `scripts/` (see README.md).

This notebook just demonstrates the core pipeline end-to-end on a tiny scale, for quick sanity-checking.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src import config
from src.patients import generate_virtual_patients, split_patients
from src.simulator import build_model_for_patient, compute_electrode_templates, render_percept
from src.encoder import StimulusEncoder
from src.training import train_encoder
from src.evaluate import evaluate_encoder

from torchvision import transforms
from torchvision.datasets import MNIST

print('Device:', config.DEVICE)

In [ ]:
# Generate a small set of virtual patients
virtual_patients = generate_virtual_patients(10, seed=0)
pretrain_patients, holdout_patients = split_patients(virtual_patients, n_pretrain=8, n_holdout=2)
print(virtual_patients[0])

In [ ]:
# Load MNIST and build a simulator for one patient
transform = transforms.Compose([transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)), transforms.ToTensor()])
mnist = MNIST(root='../data', train=True, download=True, transform=transform)

implant, model = build_model_for_patient(pretrain_patients[0])
template = compute_electrode_templates(implant, model)
print(f'{len(implant.electrode_names)} electrodes')

In [ ]:
# Quick train (small step count, for demo purposes only)
encoder = StimulusEncoder(n_electrodes=len(implant.electrode_names))
encoder, losses = train_encoder(encoder, mnist, list(range(100)), template, n_steps=30)
print('Final loss:', losses[-1])

In [ ]:
# Evaluate
eval_idx = list(range(500, 520))
ssim_mean, ssim_sd = evaluate_encoder(encoder, mnist, implant, model, eval_idx)
print(f'SSIM: {ssim_mean:.4f} +/- {ssim_sd:.4f}')